In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

print("1. Loading raw dataset...")
# Load the 5-year Kaggle dataset (913,000 rows)
df = pd.read_csv('train.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(by=['store', 'item', 'date'])

# ---------------------------------------------------------
# 2. CALENDAR & SEASONALITY DIMENSION (Pages 2 & 9)
# ---------------------------------------------------------
print("2. Engineering Calendar & Seasonality dimensions...")
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['quarter'] = df['date'].dt.quarter
df['day_of_week'] = df['date'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

# Map Months to Indian Seasons for NorthBay Living
def get_season(month):
    if month in [12, 1, 2]: return 'Winter'
    elif month in [3, 4, 5]: return 'Summer'
    elif month in [6, 7, 8, 9]: return 'Monsoon'
    else: return 'Autumn'
df['season'] = df['month'].apply(get_season)

# ---------------------------------------------------------
# 3. STORE & SKU MASTER DIMENSIONS (Pages 3 & 4)
# ---------------------------------------------------------
print("3. Generating Store Regions and SKU Categories...")
np.random.seed(42)

# Map Stores to Regions
regions = {1:'North', 2:'South', 3:'East', 4:'West', 5:'Central',
           6:'North', 7:'South', 8:'East', 9:'West', 10:'Central'}
df['region'] = df['store'].map(regions)

# Synthesize SKU Master (Categories, Subcategories, Financials)
unique_items = df['item'].unique()
categories = ['Home Decor', 'Kitchen & Dining', 'Furniture', 'Small Appliances', 'Bedding & Bath']
subcategories = ['Premium', 'Standard', 'Budget']

sku_data = []
for item in unique_items:
    cat = np.random.choice(categories)
    subcat = np.random.choice(subcategories)
    unit_cost = np.random.randint(500, 3000) # Rupees
    list_price = int(unit_cost * np.random.uniform(1.3, 2.0)) # 30% to 100% Markup
    sku_data.append({'item': item, 'category': cat, 'subcategory': subcat,
                     'unit_cost': unit_cost, 'list_price': list_price})

sku_master = pd.DataFrame(sku_data)
df = df.merge(sku_master, on='item', how='left')

# ---------------------------------------------------------
# 4. PROMOTIONS & FINANCIALS (Pages 1 & 8)
# ---------------------------------------------------------
print("4. Applying Promotional Logic and calculating Profit...")
# Simulate random promotional events
promo_events = ['Diwali Sale', 'Summer Clearance', 'End of Year', 'None']
df['promo_event'] = np.random.choice(promo_events, p=[0.05, 0.05, 0.05, 0.85], size=len(df))
df['promo_flag'] = np.where(df['promo_event'] != 'None', 1, 0)

# If on promo, apply a 10% to 30% discount
df['discount_pct'] = np.where(df['promo_flag'] == 1, np.random.uniform(0.10, 0.30, len(df)), 0.0)
df['selling_price'] = df['list_price'] * (1 - df['discount_pct'])

# Calculate Core KPIs
df['revenue'] = df['sales'] * df['selling_price']
df['cost'] = df['sales'] * df['unit_cost']
df['profit'] = df['revenue'] - df['cost']

# ---------------------------------------------------------
# 5. INVENTORY SNAPSHOTS (Pages 5, 6, & 7)
# ---------------------------------------------------------
print("5. Generating dynamic Inventory constraints...")
# Reorder points based on average sales momentum
avg_sales = df.groupby(['store', 'item'])['sales'].transform('mean')
df['reorder_point'] = (avg_sales * 7).astype(int)
df['lead_time_days'] = np.random.choice([3, 5, 7], size=len(df))

# Simulate On-Hand Units (A proxy that fluctuates around the average sales volume)
df['on_hand_units'] = (df['sales'] * 3) + np.random.randint(-15, 60, size=len(df))
df['on_hand_units'] = df['on_hand_units'].clip(lower=0) # Can't have negative inventory
df['inventory_value'] = df['on_hand_units'] * df['unit_cost']

# ---------------------------------------------------------
# 6. FEATURE ENGINEERING & MODELING (Page 10)
# ---------------------------------------------------------
print("6. Training forecasting model (LightGBM)...")
df['sales_lag_7'] = df.groupby(['store', 'item'])['sales'].shift(7)
df['rolling_mean_7'] = df.groupby(['store', 'item'])['sales'].transform(lambda x: x.shift(1).rolling(window=7).mean())
df['rolling_mean_30'] = df.groupby(['store', 'item'])['sales'].transform(lambda x: x.shift(1).rolling(window=30).mean())

df = df.dropna().copy() # Drop NaNs created by 30-day shift

# Train on 2013-2016, Predict on 2017
train = df[df['year'] < 2017]
test = df[df['year'] >= 2017]

features = ['store', 'item', 'month', 'day_of_week', 'is_weekend', 'promo_flag', 'sales_lag_7', 'rolling_mean_7', 'rolling_mean_30']
target = 'sales'

model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, random_state=42, n_jobs=-1)
model.fit(train[features], train[target])

# Apply predictions back to the entire DataFrame so Power BI has history
df['model_forecast'] = model.predict(df[features]).clip(min=0)
df['baseline_forecast'] = df['sales_lag_7'] # Seasonal-Naive Baseline

# ---------------------------------------------------------
# 7. RISK SCORING & BUSINESS IMPACT (Pages 11 & 12)
# ---------------------------------------------------------
print("7. Calculating Risk Quadrants and Rupee Impact...")
df['7_day_forecast'] = df['model_forecast'] * 7

# Thresholds defined by the Client Brief
df['stockout_risk'] = np.where(df['on_hand_units'] < df['7_day_forecast'], 1, 0)
df['overstock_risk'] = np.where(df['on_hand_units'] > (df['model_forecast'] * 28), 1, 0) # Over 4 weeks of stock

def assign_quadrant(row):
    if row['stockout_risk'] == 1 and row['overstock_risk'] == 0: return 'Reorder Now'
    elif row['overstock_risk'] == 1 and row['stockout_risk'] == 0: return 'Markdown / Clear'
    elif row['stockout_risk'] == 1 and row['overstock_risk'] == 1: return 'Watch / Volatile'
    else: return 'Healthy'

df['decision_action'] = df.apply(assign_quadrant, axis=1)

# Rupee Impact Calculations
df['stockout_rupee_impact'] = (df['7_day_forecast'] - df['on_hand_units']).clip(lower=0) * df['selling_price']
df['overstock_rupee_impact'] = (df['on_hand_units'] - (df['model_forecast'] * 28)).clip(lower=0) * df['unit_cost']
df['total_rupee_impact'] = df['stockout_rupee_impact'] + df['overstock_rupee_impact']

# ---------------------------------------------------------
# 8. FINAL EXPORT FOR POWER BI (Corrected)
# ---------------------------------------------------------
print("8. Exporting 5-year Master Dataset...")
# Reorder columns logically for Power BI, now including rolling means
export_cols = [
    'date', 'year', 'quarter', 'month', 'season', 'day_of_week', 'is_weekend',
    'store', 'region', 'item', 'category', 'subcategory',
    'unit_cost', 'list_price', 'discount_pct', 'selling_price', 'promo_flag', 'promo_event',
    'sales', 'revenue', 'cost', 'profit',
    'on_hand_units', 'reorder_point', 'lead_time_days', 'inventory_value',
    'sales_lag_7', 'rolling_mean_7', 'rolling_mean_30',  # <-- Added missing features here
    'baseline_forecast', 'model_forecast', '7_day_forecast',
    'decision_action', 'stockout_rupee_impact', 'overstock_rupee_impact', 'total_rupee_impact'
]

final_df = df[export_cols]
final_df.to_csv("Zidio_NorthBay_Master_Dataset.csv", index=False)
print(f"SUCCESS! Exported {len(final_df)} rows and {len(final_df.columns)} columns to 'Zidio_NorthBay_Master_Dataset.csv'.")

1. Loading raw dataset...
2. Engineering Calendar & Seasonality dimensions...
3. Generating Store Regions and SKU Categories...
4. Applying Promotional Logic and calculating Profit...
5. Generating dynamic Inventory constraints...
6. Training forecasting model (LightGBM)...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.035874 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 781
[LightGBM] [Info] Number of data points in the train set: 715500, number of used features: 9
[LightGBM] [Info] Start training from score 51.056034
7. Calculating Risk Quadrants and Rupee Impact...
8. Exporting 5-year Master Dataset...
SUCCESS! Exported 898000 rows and 36 columns to 'Zidio_NorthBay_Master_Dataset.csv'.
